# Mini Project: Sales Data Cleaning with Pandas

**Dataset columns:** OrderID, CustomerName, City, Product, Quantity, UnitPrice, OrderDate, OrderStatus, CustomerAge

**Goal:** find and fix missing values, outliers, data-type errors, inconsistent text, extra spaces, duplicates and date problems, add a `TotalAmount` column, and export `cleaned_sales_dataset.csv`.

> **Before you run:** upload your CSV to Colab (folder icon on the left) and set `FILE_NAME` in the next cell to its exact name. If the file isn't found, the notebook will open an upload prompt instead.

## 1. Load the data

In [ ]:
import re
import numpy as np
import pandas as pd

FILE_NAME = "messy_sales_dataset.csv"   # <-- change to your CSV's name

import os
if not os.path.exists(FILE_NAME):
    try:
        from google.colab import files
        uploaded = files.upload()
        FILE_NAME = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{FILE_NAME} not found")

# dtype=str keeps every value exactly as written, so bad entries like "three" or "18,500" aren't silently changed
raw = pd.read_csv(FILE_NAME, dtype=str, keep_default_na=True)
df = raw.copy()
raw.head(10)

## 2. Inspect the dataset

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

In [ ]:
print("Missing values per column:")
df.isnull().sum()

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print()
for col in ["City", "Product", "OrderStatus"]:
    print(f"--- {col} (raw unique values) ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

**What the inspection shows**

- Because everything was loaded as text, `info()` reports every column as `object`; the numeric and date columns need converting.
- `isnull().sum()` shows gaps in several columns (names, cities, quantities, prices, dates, ages).
- The raw `value_counts()` reveal the same city/status written in several ways (`delhi`, `DELHI`, `Delhi`) and values padded with spaces.
- Some rows are exact duplicates.

## 3. Clean text: extra spaces and consistent capitalisation

In [ ]:
text_cols = ["CustomerName", "City", "Product", "OrderStatus"]

for col in text_cols:
    df[col] = (
        df[col]
        .str.strip()                          # leading / trailing spaces
        .str.replace(r"\s+", " ", regex=True)  # repeated inner spaces
        .str.title()                          # delhi / DELHI -> Delhi
    )

for col in ["City", "Product", "OrderStatus"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

## 4. Fix data-type errors in numeric columns

Entries such as `"three"`, `"2 pcs"`, `"18,500"`, `"forty"` and `"unknown"` stop a column from being numeric. Rules used:

- number words → digits (`three` → 3, `forty` → 40)
- remove thousands separators and units (`18,500` → 18500, `2 pcs` → 2)
- anything that is not a number (`unknown`) → missing (`NaN`), to be handled in the missing-value step

In [ ]:
WORDS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
    "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
    "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
    "hundred": 100,
}

def to_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(",", "")
    if s in WORDS:
        return float(WORDS[s])
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group()) if m else np.nan

num_cols = ["Quantity", "UnitPrice", "CustomerAge"]
bad_before = {}
for col in num_cols:
    original = df[col].copy()
    df[col] = original.map(to_number)
    # entries that were present but needed fixing or became NaN
    changed = original.notna() & (original.str.strip() != df[col].map(lambda v: "" if pd.isna(v) else str(int(v)) if float(v).is_integer() else str(v)))
    bad_before[col] = original[changed].unique().tolist()

df["OrderID"] = pd.to_numeric(df["OrderID"], errors="coerce")

print("Non-numeric / malformed entries found and converted:")
for col, vals in bad_before.items():
    print(f"  {col}: {vals}")
print()
df[num_cols].describe()

## 5. Fix date inconsistencies

The dates use several formats and at least one is impossible (e.g. 30 Feb). Each value is tried against a list of known formats; day-first is assumed for slash/dash dates (Indian convention). Values that match none of them are treated as invalid and become missing (`NaT`).

In [ ]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d", "%b %d, %Y", "%B %d, %Y",
                "%d %b %Y", "%d %B %Y", "%d.%m.%Y", "%m/%d/%Y"]

def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

date_text = df["OrderDate"].copy()
df["OrderDate"] = date_text.map(parse_date)

invalid_dates = date_text[date_text.notna() & df["OrderDate"].isna()]
print("Invalid / unparseable dates:", invalid_dates.tolist())
print("Date range:", df["OrderDate"].min().date(), "to", df["OrderDate"].max().date())

## 6. Remove duplicate records

Duplicates are removed **after** the text and date clean-up, because rows such as `" Delhi "` and `"Delhi"` only look identical once standardised.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows ({before} -> {len(df)})")

# OrderID should be unique; check for any remaining repeats
print("Repeated OrderIDs remaining:", df["OrderID"].duplicated().sum())

## 7. Handle outliers

| Column | How outliers are detected | Treatment |
| --- | --- | --- |
| `CustomerAge` | Domain rule: valid ages are 18-100 | Out-of-range values → missing, then imputed |
| `UnitPrice` | Extreme IQR fence (Q3 + 3×IQR, Q1 − 3×IQR) **within each product**, plus prices ≤ 0 | Flagged → missing, then imputed |
| `Quantity` | Extreme IQR fence (3×IQR) and quantity ≤ 0 | Flagged → missing, then imputed |

Prices are compared within each product because a laptop is legitimately much more expensive than a mouse. The wider 3×IQR multiplier flags only *extreme* values, so genuine high-but-plausible orders are kept. Outliers are turned into missing values and handled together with the other gaps, rather than dropping whole rows, so no other information in those orders is lost.

In [ ]:
def iqr_fences(s, k=3.0):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

log = {}

# Age: domain rule
age_bad = df["CustomerAge"].notna() & ~df["CustomerAge"].between(18, 100)
log["CustomerAge outliers"] = df.loc[age_bad, "CustomerAge"].tolist()
df.loc[age_bad, "CustomerAge"] = np.nan

# Price: per-product IQR fence
price_bad = pd.Series(False, index=df.index)
for prod, grp in df.groupby("Product"):
    s = grp["UnitPrice"].dropna()
    if len(s) < 4:
        continue
    lo, hi = iqr_fences(s)
    price_bad.loc[grp.index] = (grp["UnitPrice"] < lo) | (grp["UnitPrice"] > hi)
price_bad |= df["UnitPrice"] <= 0
log["UnitPrice outliers"] = df.loc[price_bad, "UnitPrice"].tolist()
df.loc[price_bad, "UnitPrice"] = np.nan

# Quantity: IQR fence + positive check
lo, hi = iqr_fences(df["Quantity"].dropna())
qty_bad = (df["Quantity"] > hi) | (df["Quantity"] <= 0) | (df["Quantity"] < lo)
log["Quantity outliers"] = df.loc[qty_bad, "Quantity"].tolist()
df.loc[qty_bad, "Quantity"] = np.nan

for k, v in log.items():
    print(f"{k}: {v}")

## 8. Handle missing values

| Column | Strategy | Reason |
| --- | --- | --- |
| `CustomerName`, `City` | Fill with `"Unknown"` | Can't be inferred reliably; keeping the order is more useful than deleting it |
| `Quantity` | Median (whole number) | Median isn't distorted by the outliers just removed |
| `UnitPrice` | Median price of the same product | Price depends mainly on the product |
| `CustomerAge` | Median age | Simple, robust central value |
| `OrderDate` | Left as missing (`NaT`) | A date can't be guessed without inventing data; flagged so it can be reviewed |
| `OrderStatus`, `Product` | Fill with `"Unknown"` if any are missing | Keeps the column complete |

In [ ]:
print("Missing values BEFORE imputation:")
print(df.isnull().sum().to_string())

for col in ["CustomerName", "City", "Product", "OrderStatus"]:
    df[col] = df[col].fillna("Unknown")

df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median()).round()

price_median_by_product = df.groupby("Product")["UnitPrice"].transform("median")
df["UnitPrice"] = df["UnitPrice"].fillna(price_median_by_product).fillna(df["UnitPrice"].median())

df["CustomerAge"] = df["CustomerAge"].fillna(df["CustomerAge"].median()).round()

print()
print("Missing values AFTER imputation:")
print(df.isnull().sum().to_string())

## 9. Convert to suitable data types

In [ ]:
df = df.astype({
    "OrderID": "int64",
    "CustomerName": "string",
    "City": "category",
    "Product": "category",
    "Quantity": "int64",
    "UnitPrice": "float64",
    "OrderStatus": "category",
    "CustomerAge": "int64",
})
# OrderDate is already datetime64[ns]
df.dtypes

## 10. Create the `TotalAmount` column

In [ ]:
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df[["OrderID", "Product", "Quantity", "UnitPrice", "TotalAmount"]].head()

## 11. Final checks

In [ ]:
print("Shape:", df.shape)
print()
df.info()
print()
print("Duplicates:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum(), "(only OrderDate may remain - see Section 8)")
print()
print("Cities:", sorted(df["City"].unique()))
print("Statuses:", sorted(df["OrderStatus"].unique()))
print()
df.describe()

## 12. Export the cleaned dataset

In [ ]:
df.to_csv("cleaned_sales_dataset.csv", index=False)
print("Saved cleaned_sales_dataset.csv with", len(df), "rows")

try:
    from google.colab import files
    files.download("cleaned_sales_dataset.csv")
except ImportError:
    pass

## 13. Summary of cleaning steps

1. **Loaded** the CSV as text so malformed values stayed visible, then inspected it with `head()`, `info()`, `shape` and `isnull().sum()`.
2. **Text:** trimmed leading/trailing and repeated spaces and converted names, cities, products and statuses to Title Case so `delhi`, `DELHI` and `Delhi` became one value.
3. **Data-type errors:** converted number words and strings like `2 pcs` and `18,500` to numbers; non-numeric entries such as `unknown` became missing.
4. **Dates:** parsed all formats into one `datetime` type; impossible dates became missing.
5. **Duplicates:** removed exact duplicate rows after standardising, so hidden duplicates were also caught.
6. **Outliers:** flagged unrealistic ages (outside 18-100), extreme prices (per product, 3×IQR) and extreme quantities (3×IQR); set them to missing rather than deleting the whole order.
7. **Missing values:** names/cities → `"Unknown"`; quantity and age → median; price → median of the same product; missing dates left as `NaT` rather than invented.
8. **Types and new column:** set suitable dtypes and added `TotalAmount = Quantity × UnitPrice`.
9. **Exported** `cleaned_sales_dataset.csv`.